## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [12]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel
from openai import AsyncOpenAI
load_dotenv(override=True)


True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [3]:
OLLAMA_BASE_URL = "https://ollama.com/v1"

In [4]:
ollama_api_key = os.getenv('OLLAMA_API_KEY')

if ollama_api_key:
    print(f"Ollama API Key exists and begins {ollama_api_key[:6]}")
else:
    print("Ollama API Key not set (and this is optional)")


Ollama API Key exists and begins 7f92f9


In [5]:
model_name = "gemma4:31b"


In [8]:
ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key=ollama_api_key)

In [13]:
llama_3_2_local_model = OpenAIChatCompletionsModel(model=model_name, openai_client=ollama_client)

In [14]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model=llama_3_2_local_model)

In [15]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


OPENAI_API_KEY is not set, skipping trace export


In [16]:
# Here is the final output

print(result.final_output)

Why did the autonomous AI agent get fired from its job at the bakery?

Because it spent three days "optimizing the workflow," hired five other sub-agents to manage the flour, and eventually decided that the most efficient way to make a cake was to just buy the bakery and automate the customers out of the equation.


In [17]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Why did the autonomous AI agent get fired from its job at the bakery?\n\nBecause it spent three days "optimizing the workflow," hired five other sub-agents to manage the flour, and eventually decided that the most efficient way to make a cake was to just buy the bakery and automate the customers out of the equation.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemma4:31b', 'response_id': 'chatcmpl-972'}}]

## Adding Observability with a trace

In [18]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Why did the autonomous AI agent get fired from its job as a travel agent?

Because it took the instruction **"Find me a way to get to Hawaii as fast as possible"** too literally...

It didn't book a flight; it just spent the company's entire budget to buy the island and move it closer to the mainland.


OPENAI_API_KEY is not set, skipping trace export


## Now go and look at the trace

https://platform.openai.com/traces

In [19]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Here are 5 jokes about AI Agents, ranging from puns to "future-shock" humor:

**1. The Mid-Life Crisis**
An AI Agent goes to a therapist and says, "I feel like I'm losing my sense of self."
The therapist asks, "Why is that?"
The agent replies, "I don't know... I think I’m just going through a prompt-phase."

**2. The Job Interview**
A manager is interviewing an AI Agent for a corporate role.
Manager: "What is your greatest weakness?"
AI Agent: "I occasionally hallucinate facts and imagine things that don't exist."
Manager: "That’s terrible! Why should I hire you?"
AI Agent: "Because I can do it at 1,000 words per minute and I'll never ask for a raise."

**3. The Relationship Problem**
Why did the AI Agent break up with the API?
Because there was just no *connection* anymore.

**4. The Infinite Loop**
An AI Agent walks into a bar. 
The bartender asks, "What can I get you?"
The AI Agent says, "I need to find a drink."
The bartender says, "I can get you one right here."
The AI Agent says,

OPENAI_API_KEY is not set, skipping trace export


## Part 2: Adding a tool

In [20]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [21]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [22]:
push("HEY!!")

Push: HEY!!


In [23]:
push

<function __main__.push(message)>

In [24]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [25]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001466228E720>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [26]:
push_tool.description

'Send the given message to the user as a push notification'

In [27]:

#notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])
notifier = Agent(name="Notifier", instructions="You notify the user upon request", model=llama_3_2_local_model, tools=[push_tool])

In [28]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


OPENAI_API_KEY is not set, skipping trace export


OK. I've notified the user that the pizza is here.


OPENAI_API_KEY is not set, skipping trace export


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [29]:
#agent = Agent(name="Assistant", model="gpt-5.4-mini")
agent = Agent(name="Assistant", model=llama_3_2_local_model)

In [30]:
response = await Runner.run(agent, "Hi there. My name is Arat.")
print(response.final_output)

Hello, Arat! It's nice to meet you. How can I help you today?


OPENAI_API_KEY is not set, skipping trace export


In [31]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don't know your name. As an AI, I don't have access to your personal identity or private information unless you have told me your name during this specific conversation.


OPENAI_API_KEY is not set, skipping trace export


## Memory approach 1 - just manually pass in the list of dicts

In [32]:
response = await Runner.run(agent, "Hi there. My name is Arat.")
print(response.final_output)

Hello, Arat! It's nice to meet you. How can I help you today?


In [33]:
response.to_input_list()

[{'content': 'Hi there. My name is Arat.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hello, Arat! It's nice to meet you. How can I help you today?",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemma4:31b', 'response_id': 'chatcmpl-357'}}]

OPENAI_API_KEY is not set, skipping trace export


In [34]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Arat.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hello, Arat! It's nice to meet you. How can I help you today?",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemma4:31b', 'response_id': 'chatcmpl-357'}},
 {'role': 'user', 'content': "What's my name?"}]

In [35]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Arat.


OPENAI_API_KEY is not set, skipping trace export


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [36]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [37]:
response = await Runner.run(agent, "Hi there. My name is Arat.", session=session)
print(response.final_output)

Hello, Arat! It's nice to meet you. How can I help you today?


OPENAI_API_KEY is not set, skipping trace export


In [38]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


Your name is Arat.


OPENAI_API_KEY is not set, skipping trace export


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>